# PA2: CNN + RNN for Human Activity & Postural-Transition Recognition (HAPT)
**CSE 5526: Intro to Neural Networks**

> **Instructions:** Only edit sections marked with `# TODO`. Do not modify any other code —
> in particular, the cells marked **PROVIDED — DO NOT MODIFY** (the data pipeline) and the
> fixed function signatures, docstrings, and print statements must stay unchanged.
> Run all cells before submitting. Unexecuted notebooks receive a zero.
> Submit this single file as **`pa2.ipynb`** to your `pa2` folder on GitHub.

## 1. Imports and setup

In [ ]:
import os, glob, re, time, copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             confusion_matrix)
%matplotlib inline

# TODO: Set all random seeds (Python/NumPy/PyTorch) for reproducibility using seed 5526
# YOUR CODE HERE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('PyTorch version:', torch.__version__)

## 2. Data preparation  — **PROVIDED — DO NOT MODIFY**

The HAPT `RawData/` folder holds one continuous recording per experiment
(`acc_expXX_userYY.txt`, `gyro_expXX_userYY.txt`, 50 Hz) and a single `labels.txt`
with rows `(experiment, user, activity, start_sample, end_sample)`.

The code below (do not change it):
1. stacks acc + gyro into a **6-channel** continuous stream per experiment and labels
   every sample from `labels.txt` (unlabeled gaps get `-1`);
2. cuts each stream into consecutive non-overlapping windows of `W = 64` samples (1.28 s);
   a window's label is its majority activity if ≥60% of it is labeled, else `-1`
   (masked out of the loss but kept in the sequence for context);
3. standardizes each channel using the **training subjects only**;
4. splits **by subject** (official HAPT test users; 3 train users held out for validation).

Set `DATA_DIR` to the folder that contains `RawData/`.

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
DATA_DIR = os.environ.get('HAPT_DIR', 'HAPT Dataset')   # <- folder containing RawData/
RAW      = os.path.join(DATA_DIR, 'RawData')

FS, W, N_CLASSES = 50, 64, 12
TEST_USERS = {2, 4, 9, 10, 12, 13, 18, 20, 24}
VAL_USERS  = {7, 22, 28}
CLASS_NAMES = ['WALKING','WALKING_UPSTAIRS','WALKING_DOWNSTAIRS','SITTING','STANDING',
               'LAYING','STAND_TO_SIT','SIT_TO_STAND','SIT_TO_LIE','LIE_TO_SIT',
               'STAND_TO_LIE','LIE_TO_STAND']
SHORT = ['WALK','W_UP','W_DN','SIT','STAND','LAY',
         'ST→SIT','SIT→ST','SIT→LIE','LIE→SIT','ST→LIE','LIE→ST']
TRANSITION = list(range(6, 12))   # transition classes (activities 7–12)

def load_label_spans():
    lab = np.loadtxt(os.path.join(RAW, 'labels.txt'), dtype=int)
    spans = {}
    for exp, user, act, s, e in lab:
        spans.setdefault(exp, []).append((act, s, e))
    return spans

def build_experiments():
    spans = load_label_spans(); exps = []
    for acc_path in sorted(glob.glob(os.path.join(RAW, 'acc_exp*.txt'))):
        m = re.search(r'acc_exp(\d+)_user(\d+)\.txt', acc_path)
        exp, user = int(m.group(1)), int(m.group(2))
        acc = np.loadtxt(acc_path, dtype=np.float32)
        gyr = np.loadtxt(acc_path.replace('acc_', 'gyro_'), dtype=np.float32)
        n = min(len(acc), len(gyr))
        sig = np.concatenate([acc[:n], gyr[:n]], axis=1)      # (n, 6)
        y = np.full(n, -1, dtype=np.int64)
        for act, s, e in spans.get(exp, []):
            y[s:e+1] = act - 1
        exps.append(dict(exp=exp, user=user, sig=sig, y=y))
    return exps

def window_stream(sig, y):
    xs, ys = [], []
    for i in range(0, len(sig) - W + 1, W):
        wy = y[i:i+W]; valid = wy[wy >= 0]; lab = -1
        if len(valid) >= 0.6 * W:
            vals, cnts = np.unique(valid, return_counts=True)
            lab = int(vals[cnts.argmax()])
        xs.append(sig[i:i+W].T); ys.append(lab)
    return np.asarray(xs, np.float32), np.asarray(ys, np.int64)

_exps = build_experiments()
_pool = np.concatenate([e['sig'] for e in _exps
                        if e['user'] not in TEST_USERS | VAL_USERS])
_mean, _std = _pool.mean(0), _pool.std(0) + 1e-8

train_seqs, val_seqs, test_seqs = [], [], []
for e in _exps:
    x, y = window_stream((e['sig'] - _mean) / _std, e['y'])
    (test_seqs if e['user'] in TEST_USERS else
     val_seqs  if e['user'] in VAL_USERS  else train_seqs).append((x, y))

def _n_labeled(seqs): return int(sum((y >= 0).sum() for _, y in seqs))
print(f'experiments: train {len(train_seqs)} | val {len(val_seqs)} | test {len(test_seqs)}')
print(f'labeled windows: train {_n_labeled(train_seqs)} | '
      f'val {_n_labeled(val_seqs)} | test {_n_labeled(test_seqs)}')
# =================================================================================

## 3. Class distribution  — **PROVIDED — DO NOT MODIFY**

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
_counts = np.zeros(N_CLASSES, int)
for _, y in train_seqs:
    for c in y[y >= 0]:
        _counts[c] += 1
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(range(N_CLASSES), _counts, color=['#4C78A8']*6 + ['#E45756']*6)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(SHORT, rotation=45, ha='right')
ax.set_ylabel('train windows'); ax.set_title('Class distribution (red = postural transitions)')
plt.tight_layout(); plt.show()
print('transitions are %.1f%% of labeled training windows' %
      (100*_counts[TRANSITION].sum()/_counts.sum()))
# =================================================================================

## 4. Model definitions

All three models share one identical CNN **window encoder**, so any performance difference
is attributable to temporal context rather than CNN capacity.

### Xavier initialization
Implement a helper that applies Xavier-uniform initialization to all weight matrices
(`Conv1d`, `Linear`, `LSTM`) and sets biases to zero.

In [ ]:
def xavier_init(module):
    '''Apply Xavier-uniform init to all weight matrices (dim >= 2); zero all biases.

    Args:
        module (nn.Module): model to initialize in place
    '''
    # TODO
    pass

### Shared window encoder
Maps one window `(6 channels × 64 time steps)` to a 64-d embedding:
- `Conv1d(6 → 32, kernel 5, padding 2)` → `BatchNorm1d(32)` → `ReLU`
- `Conv1d(32 → 64, kernel 5, padding 2)` → `BatchNorm1d(64)` → `ReLU`
- `AdaptiveAvgPool1d(1)` → squeeze time → `(N, 64)`

In [ ]:
class Encoder(nn.Module):
    '''Shared per-window CNN encoder: input (Nw, 6, W) -> output (Nw, emb).'''
    def __init__(self, in_ch=6, emb=64):
        super().__init__()
        # TODO: define self.net (nn.Sequential) per the spec above
        self.net = None  # TODO
        self.emb = emb

    def forward(self, w):
        '''w: (Nw, 6, W)  ->  (Nw, emb)'''
        # TODO
        pass

### Model 1 — CNN-only (no temporal context)
`encoder → Dropout(0.3) → Linear(64 → 12)`. Each window classified independently.

In [ ]:
class CNNOnly(nn.Module):
    def __init__(self, emb=64, dropout=0.3):
        super().__init__()
        self.enc = Encoder(emb=emb)
        # TODO: define self.drop and self.fc (Linear emb -> N_CLASSES)
        self.drop = None  # TODO
        self.fc = None    # TODO
        xavier_init(self)

    def forward(self, w):
        '''w: (Nw, 6, W) -> logits (Nw, N_CLASSES)'''
        # TODO
        pass

_m = CNNOnly(); print('CNN-only params:',
                      sum(p.numel() for p in _m.parameters() if p.requires_grad))

### Models 2 & 3 — CNN + LSTM and CNN + Bi-LSTM
Encode every window in a recording, then run an LSTM across the window sequence
(**many-to-many**: one prediction per window).
- `LSTM(input 64, hidden 128, 1 layer, batch_first=True, bidirectional=bidir)`
- `Dropout(0.3) → Linear(128·dir → 12)`  (dir = 2 if bidirectional else 1)

Set `bidir=False` for the CNN+LSTM, `bidir=True` for the CNN+Bi-LSTM.

In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self, emb=64, hidden=128, layers=1, bidir=False, dropout=0.3):
        super().__init__()
        self.enc = Encoder(emb=emb)
        # TODO: define self.lstm, self.drop, self.fc per the spec above
        self.lstm = None  # TODO
        self.drop = None  # TODO
        self.fc = None    # TODO
        xavier_init(self)

    def forward(self, w):
        '''w: one recording's windows (Nw, 6, W) -> logits (Nw, N_CLASSES).
        Hint: encode windows -> (1, Nw, emb), run the LSTM, then the linear head.'''
        # TODO
        pass

for _name, _mm in [('CNN+LSTM', CNNLSTM(bidir=False)), ('CNN+BiLSTM', CNNLSTM(bidir=True))]:
    print(f'{_name} params:', sum(p.numel() for p in _mm.parameters() if p.requires_grad))

## 5. Training and evaluation functions

Implement the functions below. **Signatures, docstrings, return values, and the per-epoch
print statement are fixed — do not change them.**

**Training configuration (do not change):**
- Xavier-uniform init; Adam, lr = 1e-3
- Loss: **class-weighted** cross-entropy (provided `weights`), `ignore_index=-1`
- Gradient clipping, max norm 5.0
- One recording's full window sequence = one training example
- Early stopping on **validation loss** (class-weighted CE), patience 6, restore best weights; max 25 epochs

### Class weights — **PROVIDED — DO NOT MODIFY**

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
def class_weights(seqs):
    counts = np.zeros(N_CLASSES)
    for _, y in seqs:
        for c in y[y >= 0]:
            counts[c] += 1
    w = counts.sum() / (N_CLASSES * np.maximum(counts, 1))
    return torch.tensor(w, dtype=torch.float32, device=device)

weights = class_weights(train_seqs)
print('class weights:', np.round(weights.cpu().numpy(), 2))
# =================================================================================

In [ ]:
def train_model(model, train_seqs, val_seqs, weights,
                max_epochs=25, patience=6, lr=1e-3, clip=5.0):
    '''Train with Adam + class-weighted CE + gradient clipping, early-stopping on
    validation loss (the class-weighted cross-entropy; restore best weights).

    One training example = one recording's full window sequence (x: (Nw,6,W), y: (Nw,)).
    Use ignore_index=-1 so masked windows do not contribute to the loss.

    Each epoch, print EXACTLY (do not change this format):
        f'  epoch {ep:2d}  train_loss {tl:.4f}  val_loss {vl:.4f}  val_macro_recall {vr:.4f}'

    Returns:
        history (dict): {'train_loss': [...], 'val_loss': [...], 'val_recall': [...]}
    '''
    # TODO
    pass

In [ ]:
def evaluate(model, seqs):
    '''Evaluate on the labeled windows of `seqs` (ignore windows with label -1).

    Returns a dict with EXACTLY these keys:
        'acc'          : overall accuracy (float)
        'macro_recall' : macro-averaged recall (float)
        'macro_prec'   : macro-averaged precision (float)
        'trans_recall' : mean recall over the 6 transition classes (float)
        'per_recall'   : per-class recall, shape (12,)
        'cm'           : confusion matrix, shape (12, 12), labels 0..11
    '''
    # TODO
    pass

In [ ]:
def plot_learning_curves(history, title=''):
    '''Two panels: (left) train vs val loss; (right) validation macro-recall.
    Include axis labels, legend, and grid.'''
    # TODO
    pass

In [ ]:
def plot_confusion_matrix(cm, class_names, title=''):
    '''Display a 12x12 confusion matrix (cmap='Blues'); rotate x-tick labels 45 deg.'''
    # TODO
    pass

print('Functions defined.')

## 6. Part I — Baseline: CNN-only

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
torch.manual_seed(5526); np.random.seed(5526)
print('Training baseline CNN-only...'); print('=' * 70)
baseline = CNNOnly()
history = train_model(baseline, train_seqs, val_seqs, weights,
                      max_epochs=25, patience=6, lr=1e-3)
plot_learning_curves(history, title='CNN-only (baseline) — learning curves')
# =================================================================================

### (1) Learning curves
> **TODO (written answer):** Does the baseline overfit, underfit, or reasonably fit the
> validation data? Justify using the curves.

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
res_base = evaluate(baseline, test_seqs)
print(f"CNN-only (baseline) test | accuracy {res_base['acc']:.4f} | "
      f"macro-recall {res_base['macro_recall']:.4f} | "
      f"transition-recall {res_base['trans_recall']:.4f}")
plot_confusion_matrix(res_base['cm'], SHORT,
                      title=f"CNN-only baseline test confusion (acc {res_base['acc']:.3f})")
# =================================================================================

## 7. Part II — Does adding recurrence help? (CNN-only vs CNN+LSTM vs CNN+Bi-LSTM)

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
results = {'CNN-only': res_base}
for name, ctor in [('CNN+LSTM', lambda: CNNLSTM(bidir=False)),
                   ('CNN+BiLSTM', lambda: CNNLSTM(bidir=True))]:
    print(f'\nTraining {name}...'); print('=' * 70)
    torch.manual_seed(5526); np.random.seed(5526)
    model = ctor()
    train_model(model, train_seqs, val_seqs, weights,
                max_epochs=25, patience=6, lr=1e-3)
    results[name] = evaluate(model, test_seqs)

order = ['CNN-only', 'CNN+LSTM', 'CNN+BiLSTM']
print('\n' + '=' * 70)
print(f"{'model':<12}{'overall_acc':>13}{'macro_recall':>14}{'transition_recall':>19}")
print('-' * 70)
for k in order:
    r = results[k]
    print(f"{k:<12}{r['acc']:>13.4f}{r['macro_recall']:>14.4f}{r['trans_recall']:>19.4f}")

print('\nPer-transition-class recall:')
print(f"{'model':<12}" + ''.join(f'{SHORT[c]:>9}' for c in TRANSITION))
for k in order:
    per = results[k]['per_recall']
    print(f"{k:<12}" + ''.join(f'{per[c]:>9.3f}' for c in TRANSITION))
# =================================================================================

### Discussion
> **TODO (written answer):** Does adding the recurrence help? *Where* is the difference
> concentrated: which classes benefit most, and why? (Consider what distinguishes
> `STAND_TO_LIE` from its reverse `LIE_TO_STAND` within a single short window.)

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
plot_confusion_matrix(results['CNN-only']['cm'], SHORT,
                      title=f"CNN-only test confusion (acc {results['CNN-only']['acc']:.3f})")
plot_confusion_matrix(results['CNN+LSTM']['cm'], SHORT,
                      title=f"CNN+LSTM test confusion (acc {results['CNN+LSTM']['acc']:.3f})")
# =================================================================================

> **TODO (written answer):** (a) How does the unidirectional LSTM compare to the
> Bi-LSTM? Is future context helpful for this task, and why or why not?
> (b) Rank the three models on the test set. Does which model is "best" depend on whether
> you rank by overall accuracy, macro-recall, or transition recall? Argue which metric is
> the appropriate basis for ranking on this imbalanced task, and state what ranking by
> overall accuracy alone would hide.

## 8. Summary

In [ ]:
# ============================ PROVIDED — DO NOT MODIFY ============================
print(f"{'model':<12}{'acc':>8}{'macro_recall':>14}{'transition_recall':>19}")
for k in ['CNN-only', 'CNN+LSTM', 'CNN+BiLSTM']:
    r = results[k]
    print(f"{k:<12}{r['acc']:>8.4f}{r['macro_recall']:>14.4f}{r['trans_recall']:>19.4f}")
# =================================================================================